# Drift-Aware Resource Prediction Pipeline
## Load Data from Google Drive + Complete Processing

**Purpose:** Load data directly from Google Drive nested folder structure and process through complete pipeline

**Data Source:** 
- https://drive.google.com/drive/folders/1HStbkR9iweNIVCjyDyJqps4mFl_9rySB

**Folder Structure:** 
```
raw/
├── complex/
│   ├── case1/container/*.csv
│   └── case2/container/*.csv
└── single/
    └── case2/container/*.csv
```

**Pipeline:**
1. Mount Google Drive
2. Load data from all nested container folders (recursively)
3. Merge metrics
4. Split & normalize (prevent leakage)
5. Feature engineering
6. Sequence generation
7. Save to local sequences folder

**Output:** Sequences ready for GRU training

---
**Author:** Team-Dracasys | **Date:** 2026-05-16

# STEP 1: Mount Google Drive & Verify Access

In [ ]:
# Check if running in Google Colab
import sys
try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("⚠ Running locally (not Google Colab)")
    print("This notebook requires Google Colab to access Google Drive")

print(f"Python version: {sys.version}")

In [ ]:
if IN_COLAB:
    from google.colab import drive
    from pathlib import Path
    
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    print("\n✓ Google Drive mounted!")
    print("\nVerifying access to raw data folder...")
    
    # Check if the folder exists
    raw_path = Path('/content/drive/My Drive/raw')
    
    if raw_path.exists():
        print(f"✓ Found raw folder at: {raw_path}")
        
        print("\nExpected folder structure:")
        print("  raw/")
        print("    ├── complex/")
        print("    │   ├── case1/container/*.csv")
        print("    │   └── case2/container/*.csv")
        print("    └── single/")
        print("        └── case2/container/*.csv")
        
        print("\nActual folder structure found:")
        for item in sorted(raw_path.iterdir()):
            if item.is_dir():
                print(f"\n  📁 {item.name}/")
                for subitem in sorted(item.iterdir()):
                    if subitem.is_dir():
                        print(f"    📁 {subitem.name}/")
                        for subsubitem in sorted(subitem.iterdir()):
                            if subsubitem.is_dir():
                                csv_files = list(subsubitem.glob('*.csv'))
                                print(f"      📁 {subsubitem.name}/ ({len(csv_files)} CSV files)")
    else:
        print(f"❌ raw folder not found at: {raw_path}")
        print("\nPlease make sure:")
        print("1. The Google Drive folder is shared with your account")
        print("2. The 'raw' folder is at: My Drive/raw")
        print("3. Folder structure: raw/complex/case{1,2}/container/*.csv")
        print("                     raw/single/case2/container/*.csv")
else:
    print("To use this notebook, please run it in Google Colab")
    print("Open in Colab: https://colab.research.google.com")

# STEP 2: Setup Imports & Configure Paths

In [ ]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path
from typing import List, Dict, Tuple
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ All imports successful")

In [ ]:
# Setup Paths for Google Drive

# Google Drive paths
gd_raw_path = Path('/content/drive/My Drive/raw')

# Local output paths (in Colab's temporary storage)
output_base = Path('/content/processed_data')
processed_path = output_base / 'processed'
merged_path = output_base / 'merged'
sequences_path = output_base / 'sequences'

# Create directories
for path in [processed_path, merged_path, sequences_path]:
    path.mkdir(parents=True, exist_ok=True)

print("📁 Path Configuration:")
print(f"  Google Drive Input:   {gd_raw_path}")
print(f"  Local Processed:      {processed_path}")
print(f"  Local Merged:         {merged_path}")
print(f"  Local Sequences:      {sequences_path}")

# Check Google Drive access
print(f"\n📊 Checking Google Drive folder structure:")
if gd_raw_path.exists():
    print(f"✓ Raw folder accessible")

    # Find all CSV files recursively using **/*.csv pattern
    all_csv_files = list(gd_raw_path.glob('**/*.csv'))
    print(f"\n✓ Found {len(all_csv_files)} total CSV files in nested structure:")

    if all_csv_files:
        # Group by folder path
        csv_by_path = {}
        for csv_file in all_csv_files:
            rel_path = csv_file.relative_to(gd_raw_path)
            folder_path = str(rel_path.parent)
            if folder_path not in csv_by_path:
                csv_by_path[folder_path] = []
            csv_by_path[folder_path].append(csv_file.name)

        # Print structure
        print("\nFolder structure with CSV counts:")
        for folder in sorted(csv_by_path.keys()):
            count = len(csv_by_path[folder])
            print(f"\n  📁 {folder}/")
            print(f"     └─ {count} CSV files")

            # Show first 2-3 file names
            for fname in sorted(csv_by_path[folder])[:3]:
                print(f"        • {fname}")
            if count > 3:
                print(f"        • ... and {count-3} more")
    else:
        print("⚠ No CSV files found in folder structure!")
else:
    print(f"❌ Raw folder not accessible at {gd_raw_path}")
    print("\nExpected structure:")
    print("  My Drive/raw/")
    print("  ├── complex/case1/container/*.csv")
    print("  ├── complex/case2/container/*.csv")
    print("  └── single/case2/container/*.csv")

# STEP 3: Load & Merge Data from Google Drive

In [ ]:
# Load all CSV files from Google Drive nested folders

def load_data_from_drive(raw_path: Path) -> pd.DataFrame:
    """Load all CSV files from Google Drive nested folder structure.
    
    Expected structure:
    raw/
      ├── complex/case1/container/*.csv
      ├── complex/case2/container/*.csv
      └── single/case2/container/*.csv
    """
    logger.info(f"Loading data from Google Drive: {raw_path}")
    
    all_dfs = []
    
    # Find ALL CSV files recursively using **/*.csv pattern
    csv_files = list(raw_path.glob('**/*.csv'))
    logger.info(f"Found {len(csv_files)} CSV files total")
    
    if not csv_files:
        logger.error("No CSV files found in folder structure!")
        return None
    
    # Load each CSV file
    for idx, csv_file in enumerate(csv_files, 1):
        try:
            # Get relative path for logging
            rel_path = csv_file.relative_to(raw_path)
            logger.info(f"[{idx}/{len(csv_files)}] Loading: {rel_path}")
            
            df = pd.read_csv(csv_file)
            logger.info(f"  Shape: {df.shape}")
            all_dfs.append(df)
        except Exception as e:
            logger.warning(f"  ⚠ Error loading {csv_file.name}: {e}")
    
    # Combine all dataframes
    if all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)
        logger.info(f"\n✓ Combined data shape: {combined_df.shape}")
        logger.info(f"  Rows: {len(combined_df):,}")
        logger.info(f"  Columns: {len(combined_df.columns)}")
        logger.info(f"\n  Column names: {list(combined_df.columns)}")
        return combined_df
    else:
        logger.error("No CSV files could be loaded!")
        return None

# Load data
df = load_data_from_drive(gd_raw_path)

if df is not None:
    print("\n✓ Data loaded successfully")
    print(f"\nData preview (first 5 rows):")
    print(df.head())
    print(f"\nData info:")
    print(f"  Total rows: {len(df):,}")
    print(f"  Total columns: {len(df.columns)}")
else:
    print("\n❌ Failed to load data - check folder structure above")

# STEP 4: Data Normalization & Split (Prevent Leakage)

In [ ]:
if df is not None:
    # Split chronologically (prevent data leakage)
    logger.info("Splitting data chronologically...")
    
    if 'timestamp' in df.columns:
        df = df.sort_values('timestamp').reset_index(drop=True)
    
    n = len(df)
    train_size = int(n * 0.6)
    val_size = int(n * 0.2)
    
    train_df = df[:train_size].copy()
    val_df = df[train_size:train_size+val_size].copy()
    test_df = df[train_size+val_size:].copy()
    
    logger.info(f"Train: {len(train_df):,} rows ({len(train_df)/n*100:.1f}%)")
    logger.info(f"Val:   {len(val_df):,} rows ({len(val_df)/n*100:.1f}%)")
    logger.info(f"Test:  {len(test_df):,} rows ({len(test_df)/n*100:.1f}%)")
    
    # Normalize each split independently (prevent leakage)
    logger.info("\nNormalizing each split independently...")
    
    def normalize_df(data: pd.DataFrame, split_name: str) -> pd.DataFrame:
        """Normalize using only this split's statistics."""
        df_norm = data.copy()
        numeric_cols = df_norm.select_dtypes(include=[np.float64, np.float32, int]).columns
        
        for col in numeric_cols:
            mean = df_norm[col].mean()
            std = df_norm[col].std()
            if std > 0:
                df_norm[col] = (df_norm[col] - mean) / std
            else:
                df_norm[col] = 0
        
        logger.info(f"  {split_name}: {len(numeric_cols)} columns normalized")
        return df_norm
    
    train_norm = normalize_df(train_df, 'train')
    val_norm = normalize_df(val_df, 'val')
    test_norm = normalize_df(test_df, 'test')
    
    print("\n✓ Data split and normalized")

# STEP 5: Feature Engineering

In [ ]:
if df is not None:
    logger.info("\n" + "="*70)
    logger.info("FEATURE ENGINEERING")
    logger.info("="*70)
    
    target_columns = [
        'container_cpu_usage_seconds_total',
        'container_memory_usage_bytes',
        'container_memory_working_set_bytes',
        'container_memory_rss'
    ]
    
    # Check which target columns exist
    available_targets = [col for col in target_columns if col in train_norm.columns]
    logger.info(f"Available target columns: {len(available_targets)}")
    for col in available_targets:
        logger.info(f"  - {col}")
    
    def engineer_features(data: pd.DataFrame, split_name: str) -> pd.DataFrame:
        """Create lag and rolling features."""
        logger.info(f"\nEngineering features for {split_name}...")
        df_feat = data.copy()
        
        # Sort by container and timestamp if columns exist
        if 'new_container_id' in df_feat.columns and 'timestamp' in df_feat.columns:
            df_feat = df_feat.sort_values(['new_container_id', 'timestamp']).reset_index(drop=True)
            
            # Feature engineering per container
            for container_id in df_feat['new_container_id'].unique():
                container_mask = df_feat['new_container_id'] == container_id
                container_indices = df_feat[container_mask].index
                
                for target_col in available_targets:
                    # Lag features
                    for lag in [1, 2, 3]:
                        feat_name = f"{target_col}_DIFF_{lag}"
                        df_feat.loc[container_indices, feat_name] = df_feat.loc[container_indices, target_col].diff(lag).fillna(0)
                    
                    # Rolling statistics
                    feat_name = f"{target_col}_ROLLING_MEAN_3"
                    df_feat.loc[container_indices, feat_name] = df_feat.loc[container_indices, target_col].rolling(3, min_periods=1).mean()
                    
                    feat_name = f"{target_col}_ROLLING_STD_3"
                    df_feat.loc[container_indices, feat_name] = df_feat.loc[container_indices, target_col].rolling(3, min_periods=1).std().fillna(0)
        
        logger.info(f"  Total columns after features: {len(df_feat)}")
        return df_feat
    
    train_feat = engineer_features(train_norm, 'train')
    val_feat = engineer_features(val_norm, 'val')
    test_feat = engineer_features(test_norm, 'test')
    
    # Save intermediate results
    train_feat.to_csv(merged_path / 'train_data_with_features.csv', index=False)
    val_feat.to_csv(merged_path / 'val_data_with_features.csv', index=False)
    test_feat.to_csv(merged_path / 'test_data_with_features.csv', index=False)
    
    print("\n✓ Features engineered and saved")

# STEP 6: Sequence Generation (Ultra Fast)

In [ ]:
class UltraFastSequenceGenerator:
    """Memory-efficient sequence generator."""

    def __init__(self, lookback_window: int = 240, max_horizon: int = 10):
        self.lookback_window = lookback_window
        self.max_horizon = max_horizon
        self.target_metrics = [
            'container_cpu_usage_seconds_total',
            'container_memory_usage_bytes',
            'container_memory_working_set_bytes',
            'container_memory_rss'
        ]
        self.feature_cols = None

    def determine_feature_columns(self, df: pd.DataFrame) -> List[str]:
        """Get numeric columns for features."""
        exclude_cols = {'timestamp', 'case_source', 'cmdb_id', 'new_container_id'}
        feature_cols = [col for col in df.columns 
                       if col not in exclude_cols 
                       and df[col].dtype in [np.float64, np.float32, int]]
        logger.info(f"Total input features: {len(feature_cols)}")
        return feature_cols

    def process_and_save_sequences(self, df: pd.DataFrame, dataset_name: str, output_dir: str) -> bool:
        """Process sequences and save to disk in batches."""
        logger.info(f"Creating sequences for {dataset_name}...")

        if self.feature_cols is None:
            self.feature_cols = self.determine_feature_columns(df)

        if 'new_container_id' in df.columns and 'timestamp' in df.columns:
            df = df.sort_values(['new_container_id', 'timestamp']).reset_index(drop=True)

        sequence_counts = {h: 0 for h in range(1, self.max_horizon + 1)}
        X_writers = {}
        y_writers = {}
        container_writers = {}

        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)

        # Determine if we have container_id column
        has_container = 'new_container_id' in df.columns
        
        if has_container:
            containers = df['new_container_id'].unique()
            logger.info(f"Processing {len(containers)} containers...")
            
            total_positions = 0
            valid_sequences = 0

            for idx, container_id in enumerate(containers):
                if (idx + 1) % 5 == 0:
                    logger.info(f"  Container {idx + 1}/{len(containers)}... (valid: {valid_sequences:,})")

                container_data = df[df['new_container_id'] == container_id].copy()
                n_rows = len(container_data)

                if n_rows < self.lookback_window + self.max_horizon:
                    continue

                for i in range(self.lookback_window, n_rows - self.max_horizon):
                    total_positions += 1
                    X = container_data.iloc[i - self.lookback_window:i][self.feature_cols].values.astype(np.float32)

                    if np.isnan(X).any():
                        continue

                    valid_sequences += 1

                    for horizon in range(1, self.max_horizon + 1):
                        target_idx = i + horizon - 1
                        target_row = container_data.iloc[target_idx]

                        y = np.array([
                            float(target_row[metric]) if not pd.isna(target_row[metric]) else np.nan
                            for metric in self.target_metrics
                            if metric in container_data.columns
                        ], dtype=np.float32)

                        if len(y) > 0 and not np.isnan(y).any():
                            if sequence_counts[horizon] == 0:
                                X_writers[horizon] = []
                                y_writers[horizon] = []
                                container_writers[horizon] = []

                            X_writers[horizon].append(X)
                            y_writers[horizon].append(y)
                            container_writers[horizon].append(container_id)
                            sequence_counts[horizon] += 1

                            if sequence_counts[horizon] % 1000 == 0:
                                self._save_batch(horizon, dataset_name, output_path, X_writers, y_writers, container_writers)
                                X_writers[horizon] = []
                                y_writers[horizon] = []
                                container_writers[horizon] = []
            
            logger.info(f"Positions: {total_positions:,} | Valid: {valid_sequences:,}")
        else:
            logger.warning("No container_id column found, skipping sequence generation")
            return False

        # Save remaining batches
        for horizon in range(1, self.max_horizon + 1):
            if sequence_counts[horizon] > 0 and len(X_writers[horizon]) > 0:
                self._save_batch(horizon, dataset_name, output_path, X_writers, y_writers, container_writers)

        # Save metadata
        for horizon in range(1, self.max_horizon + 1):
            if sequence_counts[horizon] > 0:
                metadata = {
                    'horizon': horizon,
                    'dataset': dataset_name,
                    'format': 'NumPy binary',
                    'n_sequences': sequence_counts[horizon],
                    'lookback_window': self.lookback_window,
                    'n_features': len(self.feature_cols),
                    'X_shape': [sequence_counts[horizon], self.lookback_window, len(self.feature_cols)],
                    'y_shape': [sequence_counts[horizon], len(y_writers[horizon][0])] if y_writers[horizon] else [0, 0],
                }
                metadata_file = output_path / f"sequences_horizon_{horizon}_metadata_{dataset_name}.json"
                with open(metadata_file, 'w') as f:
                    json.dump(metadata, f, indent=2)
                logger.info(f"Horizon {horizon}: {sequence_counts[horizon]:,} sequences")

        return True

    def _save_batch(self, horizon, dataset_name, output_path, X_writers, y_writers, container_writers):
        """Save batch to disk."""
        if len(X_writers[horizon]) == 0:
            return

        X_batch = np.array(X_writers[horizon])
        y_batch = np.array(y_writers[horizon])

        X_file = output_path / f"sequences_horizon_{horizon}_X_{dataset_name}.npy"
        y_file = output_path / f"sequences_horizon_{horizon}_y_{dataset_name}.npy"
        container_file = output_path / f"sequences_horizon_{horizon}_containers_{dataset_name}.npy"

        if X_file.exists():
            X_existing = np.load(X_file)
            X_combined = np.vstack([X_existing, X_batch])
            np.save(X_file, X_combined)
            
            y_existing = np.load(y_file)
            y_combined = np.vstack([y_existing, y_batch])
            np.save(y_file, y_combined)
        else:
            np.save(X_file, X_batch)
            np.save(y_file, y_batch)
            np.save(container_file, np.array(container_writers[horizon], dtype=object))

print("✓ SequenceGenerator class ready")

In [ ]:
if df is not None and train_feat is not None:
    logger.info("\n" + "#"*70)
    logger.info("PHASE 1: SEQUENCE GENERATION")
    logger.info("#"*70)
    
    generator = UltraFastSequenceGenerator(lookback_window=240, max_horizon=10)
    
    # Process train
    logger.info(f"\nProcessing TRAIN data...")
    train_success = generator.process_and_save_sequences(train_feat, 'train', str(sequences_path))
    
    # Process val
    logger.info(f"\nProcessing VAL data...")
    val_success = generator.process_and_save_sequences(val_feat, 'val', str(sequences_path))
    
    # Process test
    logger.info(f"\nProcessing TEST data...")
    test_success = generator.process_and_save_sequences(test_feat, 'test', str(sequences_path))
    
    if train_success and val_success and test_success:
        print("\n✓ All sequences generated successfully")
    else:
        print("\n⚠ Some sequences may not have been generated")

# STEP 7: Verify & Load Sequences

In [ ]:
# Verify generated sequences
print("="*70)
print("VERIFYING SEQUENCES")
print("="*70)

try:
    # Load horizon 1 data
    X_train = np.load(sequences_path / 'sequences_horizon_1_X_train.npy')
    y_train = np.load(sequences_path / 'sequences_horizon_1_y_train.npy')
    
    print(f"\nTraining Data (Horizon 1):")
    print(f"  X shape: {X_train.shape}")
    print(f"    - Sequences: {X_train.shape[0]:,}")
    print(f"    - Timesteps: {X_train.shape[1]}")
    print(f"    - Features: {X_train.shape[2]}")
    print(f"  y shape: {y_train.shape}")
    print(f"    - Sequences: {y_train.shape[0]:,}")
    print(f"    - Metrics: {y_train.shape[1]}")
    
    # Load val and test
    X_val = np.load(sequences_path / 'sequences_horizon_1_X_val.npy')
    X_test = np.load(sequences_path / 'sequences_horizon_1_X_test.npy')
    
    print(f"\n  Validation: {X_val.shape[0]:,} sequences")
    print(f"  Test: {X_test.shape[0]:,} sequences")
    
    # Count total files
    npy_files = list(sequences_path.glob('*.npy'))
    json_files = list(sequences_path.glob('*.json'))
    
    print(f"\n  Total .npy files: {len(npy_files)}")
    print(f"  Total .json files: {len(json_files)}")
    
    print("\n" + "="*70)
    print("✅ SEQUENCES LOADED SUCCESSFULLY!")
    print("="*70)
    print("\nReady for Phase 2: GRU Model Training")
    
except FileNotFoundError as e:
    print(f"\n❌ Error: {e}")
    print("Sequences may not have been generated. Check the logs above.")

# STEP 8: Download Sequences (Optional)

In [ ]:
# Optional: Prepare sequences for download or Google Drive upload
print("Sequences are ready in:")
print(f"  {sequences_path}")

print("\nTo download sequences in Colab:")
print("""
from google.colab import files

# Create zip of sequences
import shutil
shutil.make_archive('/tmp/sequences', 'zip', sequences_path)

# Download
files.download('/tmp/sequences.zip')
""")

print("\nTo save sequences to Google Drive:")
print("""
# Copy to Google Drive
import shutil
gd_sequences = Path('/content/drive/My Drive/sequences')
gd_sequences.mkdir(exist_ok=True)
for file in sequences_path.glob('*'):
    shutil.copy(file, gd_sequences / file.name)
""")

# Summary

## ✅ Pipeline Complete!

### What was done:
1. ✅ Mounted Google Drive
2. ✅ Loaded data from all nested container folders
   - raw/complex/case1/container/*.csv
   - raw/complex/case2/container/*.csv
   - raw/single/case2/container/*.csv
3. ✅ Split chronologically (prevent leakage)
4. ✅ Normalized each split independently
5. ✅ Engineered temporal features
6. ✅ Generated sequences (240-step lookback, 10 horizons)
7. ✅ Saved sequences in NumPy format

### Output:
- Location: `/content/processed_data/sequences/`
- Files: 30+ (10 horizons × 3 datasets × X/y/metadata)
- Format: NumPy binary (.npy)
- Ready for: GRU model training

### Next Steps:
1. Load sequences with `np.load()`
2. Build GRU model with TensorFlow
3. Train on loaded sequences
4. Evaluate and make predictions

---

**Total runtime: ~10-15 minutes** (depending on data size)

**Status: ✅ Ready for Phase 2 (GRU Training)**